# Assignment 4: Evaluating Search Engines

For this assignment, we leave aside the code we developed so far, and look into the more general issue of how to evaluate and compare different search engines. The ultimate test for any Information Retrieval system is how well it is able to satisfy the information needs of users.

# Cohen's Kappa

Our evaluation will involve the calculation of [Cohen's kappa](https://en.wikipedia.org/wiki/Cohen's_kappa) to quantify the degree to which two human assessors agree or disagree on whether results are considered relevant or not. To calculate Cohen's kappa, we are going to use the [scikit-learn library](http://scikit-learn.org/stable/):

In [2]:
! pip install --user scikit-learn


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\Admin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
from sklearn.metrics import cohen_kappa_score

This library expects relevance assessments as lists of elements where `1` stands for _relevant_ and `0` stands for _not relevant_, for example like this:

In [4]:
a1=[1,0,1,0,1,0,1,0]


This list means that the first document was assessed to be relevant, the second to be not relevant, the third to be relevant etc.

We need two assessments in order to calculate Cohen's kappa, so let's make another exemplary list that only differs on the last element:

In [5]:
a2=[1,0,1,0,1,0,1,1]


We can now invoke the library as follows to calculate the agreement between the two:

In [6]:
cohen_kappa_score(a1, a2)

0.75

This value represents high agreement. We can reach maximal agreement if the two assessments are identical:

In [7]:
cohen_kappa_score(a1, a1)

1.0

Now, let's see what happens for a third assessment that differs on three positions with the first one (the three last positions):

In [8]:
a3=[1,0,1,0,1,1,0,1]

cohen_kappa_score(a1, a3)

0.25

We get a smaller but still positive value, because these two assessments still mostly agree. If we make a further example that differs on 6 of the 8 positions, we get the following result:

In [9]:
a4=[1,0,0,1,0,1,0,1]

cohen_kappa_score(a1, a4)

-0.5

The score is now negative, because the two differ on more positions than they agree. The agreement is in fact less than what you would expect to occur just by chance. We get the maximal disagreement if we define a fifth example that disagrees on all positions:

In [10]:
a5=[0,1,0,1,0,1,0,1]

cohen_kappa_score(a1, a5)

-1.0

Be aware that the kappa score cannot be calculated if you have only `1`s or only `0`s:

In [11]:
a6=[1,1,1,1,1,1,1,1]
a7=[1,1,1,1,1,1,1,1]

cohen_kappa_score(a6, a7)

C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


nan

And in the case of a highly skewed set (either vast majority of agreements on `1` or vast majority of agreements on `0`), the kappa score can be counter-intuitive:

In [12]:
a8=[1,1,1,1,1,1,0,1]
a9=[1,1,1,1,1,1,1,0]

cohen_kappa_score(a8, a9)

-0.1428571428571428

Now that we understand how this function works, we will apply it below for our specific evaluation.

# Results and Assessments

Next, we will define some auxilary code to deal with lists of URLs from search engines and associated relevance assessments. We will encode result lists like this:

In [13]:
urls = [
    'https://en.wikipedia.org/wiki/Information_retrieval/',  # 1st result
    'http://www.dictionary.com/browse/information',          # 2nd result
    'https://nlp.stanford.edu/IR-book/'                      # ...
]

And we represent corresponding assessments, as above, as lists of the same size containing relevance values:

In [14]:
my_assessment = [1, 0, 1]
another_assessment = [0, 0, 1]

In order to nicely display URL lists, with or without related assessments, we define a function called `display_results`:

In [15]:
from IPython.display import display, HTML

def display_results(urls, assessment1=None, assessment2=None):
    lines = []
    lines.append('<table>')
    header = '<tr><th>#</th><th>Result URL</th>'
    if (assessment1):
        header += '<th>Assessment 1</th>'
    if (assessment2):
        header += '<th>Assessment 2</th>'
    header += '</tr>'
    lines.append(header)
    i = 0
    for url in urls:
        show_url = url
        if (len(url) > 80):
            show_url = url[:75] + '...'
        line = '<tr><td>{}</td><td><a href="{:s}">{:s}</a></td>'.format(i+1, url, show_url)
        if (assessment1):
            if (assessment1[i] == 0):
                line += '<td><em>Not relevant</em></td>'
            else:
                line += '<td><strong>Relevant</strong></td>'
        if (assessment2):
            if (assessment2[i] == 0):
                line += '<td><em>Not relevant</em></td>'
            else:
                line += '<td><strong>Relevant</strong></td>'
        line += '</tr>'
        lines.append(line)
        i = i+1
    lines.append('</table>')
    display( HTML(''.join(lines)) )

We can use this function to display a list of URLs, optionally together with one or two assessment lists:

In [16]:
print("Just a list of URLs:")
display_results(urls)

print("With one assessment:")
display_results(urls, my_assessment)

print("With two assessments:")
display_results(urls, my_assessment, another_assessment)

Just a list of URLs:


#,Result URL
1,https://en.wikipedia.org/wiki/Information_retrieval/
2,http://www.dictionary.com/browse/information
3,https://nlp.stanford.edu/IR-book/


With one assessment:


#,Result URL,Assessment 1
1,https://en.wikipedia.org/wiki/Information_retrieval/,Relevant
2,http://www.dictionary.com/browse/information,Not relevant
3,https://nlp.stanford.edu/IR-book/,Relevant


With two assessments:


#,Result URL,Assessment 1,Assessment 2
1,https://en.wikipedia.org/wiki/Information_retrieval/,Relevant,Not relevant
2,http://www.dictionary.com/browse/information,Not relevant,Not relevant
3,https://nlp.stanford.edu/IR-book/,Relevant,Relevant


Now we are ready to perform an actual evaluation, which will involve a substantial amount of manual work.

---

# Tasks

**Your name:** Zayne Melendez

### Task 1

Think up and formulate a information need (for example in the field of Computer Science or Medicine) for which you think the answer can be found in scientific publications. On page 152 in the book an example of such an information need is shown: "Information on whether drinking red wine is more effective at reducing the risk of heart attacks than white wine."

**Answer:** 
Information on how dark-mode UI design affects reading comprehension and visual fatigue during prolonged screen use.

Next, write down specifically what documents have to look like to satisfy your information need. For example if your information need is about finding an overview of different cancer types, you could state that a document would need to list at least ten types of cancer to satisfy your information need (among other criteria). Write this down as a protocol with rules and examples. For example, such a protocol could state that at least three out of five given criteria have to be fulfilled for a document to be considered relevant for the information need, and then specify the criteria. Or your protocol could have the form of a sequence of rules, where each rule lets you either label the document as relevant or not relevant, or proceed with the next rule. Such rules and criteria can, for example, be about the general topic of the paper, the concepts mentioned in it, the covered relations between concepts, the type of publication (research paper, overview paper, etc.), the number of references, the types of contained diagrams, and so on, depending on your specified information need.

**Answer:**

Rules:
1. The document must explicitly discuss dark mode, low-luminance interfaces, or color-contrast schemes (light-on-dark vs. dark-on-light).
2. The document must include reading comprehension metrics, such as accuracy, recall, reading speed, or comprehension tests.
3. The document must assess visual fatigue, eye strain, visual comfort, blink rate, or related physiological measures.
4. The document should involve extended or sustained viewing sessions, typically defined as 20+ minutes of continuous reading or screen interaction.
5. The document must contain experimental, qualitative, or quantative user-study data (not just theoretical discussion).
6. The paper must evaluate a relationship, such as how dark mode influences:
    
    a. reading comprehension
    
    b. eye-strain levels
    
    c. visual comfort
    
    d. task performance

The document must **not** be one where for the study (Exclusion Rules):
1. It focuses on night-shift color temperatures (blue-light filtering) rather than dark vs. light UI modes.
2. It studies paper-based reading rather than digital screens.
3. It exclusively investigates aesthetic preference with no performance/comprehension/fatigue measures.

A document is relevant if it follows rule 1, and at least 2 of the other following rules. The document is also considered irrelevent if it it fulfills any of the exclusion rules.



### Task 2

Formulate a keyword query that represents the information need. For the example on page 152 in the book (see above), the example query "wine AND red AND white AND heart AND attack AND effective" is given. (You don't need to use connectors like "AND", but if you do, make first sure your chosen search engines below actually support them.)

**Answer:** My keyword query is...

`"dark mode" AND "reading comprehension" AND ( "visual fatigue" OR "eye strain" ) AND screen`

Then submit your query to **two** of the following academic search engines:

- [Google Scholar](https://scholar.google.com) (all science disciplines)
- [Semantic Scholar](https://www.semanticscholar.org) (all science disciplines)
- [PubMed Search](https://www.ncbi.nlm.nih.gov/pubmed) (Life Sciences / biomedicine)

The right choice of two from the three search engine depends on the topic of your information need. If your information need is in the Life Sciences and biomedicine, it's probably best to include PubMed Search, but otherwise you should pick Google Scholar and Semantic Scholar.

Extract a list of the top 10 URLs of the lists of each of the search engines given the query. To be ensure that your results are reproducible, it is advised to use the private mode of your browser. Try to access the resulting publications. For the publications where that is not possible (because of dead links or because the publication is pay-walled even within the VU network), exclude them from the list and add more publications to the end of your list (that is, append results number 11, then 12, etc. to ensure you have two lists of 10 publications each). In order to deal with paywalls, you should try accessing the articles from the VU network, use
[UBVU Off-Campus
Access](https://login.vu-nl.idm.oclc.org/login), or try to find the respective documents from alternative sources (Google Scholar, for example, is very good at finding free PDFs of articles). If you get fewer than 10 results for one of the search engines, modify the keyword query above to make it more inclusive, and then redo the steps of this task.

Store your two lists of URLs in the form of Python lists as introduced above. Then, use the `display_results` function to nicely display them.

In [19]:
# URLs found, ordered from 1st to 10th
googlescholarurl = [
    'https://www.theseus.fi/bitstream/handle/10024/896088/Laine_Jere.pdf?sequence=2',
    'https://www.mdpi.com/1660-4601/22/4/609',
    'https://personales.upv.es/thinkmind/dl/conferences/achi/achi_2024/achi_2024_3_150_20069.pdf',
    'https://publications.aston.ac.uk/id/eprint/47842/',
    'https://www.tandfonline.com/doi/abs/10.1080/10447318.2024.2305982',
    'https://repository.stcloudstate.edu/edad_etds/64/',
    'https://www.utupub.fi/bitstream/handle/10024/176173/Virtanen_Julius_opinnayte.pdf?sequence=1',
    'https://ieeexplore.ieee.org/abstract/document/9089640',
    'https://jhsmr.org/index.php/jhsmr/article/view/1095',
    'https://link.springer.com/chapter/10.1007/978-3-031-35392-5_10'
       ]
semanticscholarurl = [
    'https://www.semanticscholar.org/paper/Immediate-Effects-of-Light-Mode-and-Dark-Mode-on-in-Sengsoon-Intaruk/ad4dbb00f48b0814e954878b0681cb7d1ea0b200',
    'https://www.semanticscholar.org/paper/DarkModeNotifier%3A-Automatic-Dark-Mode-Enabler-for-Kunjir-Bhandekar/9fa9f99d272c01eb59c9a22c4d08a5d0231a1b93',
    'https://www.semanticscholar.org/paper/An-Exploration-of-Effects-of-Dark-Mode-on-Students%3A-Shrestha-Shrestha/66a46cf7d8162b490d4717772e40c03d8d375677',
    'https://www.semanticscholar.org/paper/The-effects-of-lutein-zeaxanthin-(Lute-gen%C2%AE)-on-eye-Lopresti-Smith/16bd90550858d4aa4147290584ded9a60503b83a',
    'https://www.semanticscholar.org/paper/Blink-Mate%3A-A-Smart-Screen-Control-Application-to-Dubey-Dubey/e5035814c3fad840fe46cf8dd0f703e327b0f4c5',
    'https://www.semanticscholar.org/paper/Interventions-to-prevent-visual-fatigue-during-Wong-Kopecny/4024bb16eb7ea16aeed838f359762ab7be3f9346',
    'https://www.semanticscholar.org/paper/Clinical-Efficacy-of-Ayurvedic-Management-in-Eye-Magar/51a66a1fc642c527b037c1c0c9033ed68778fb0e',
    'https://www.semanticscholar.org/paper/Astaxanthin-(AstaReal%C2%AE)-Improved-Acute-and-Chronic-Hecht-Marwah/520017cfabc17d339f4b99295675672c21a0efed',
    'https://www.semanticscholar.org/paper/THE-EFFECTS-OF-BLUE-LIGHT-BLOCKING-GLASSES-VERSUS-Saleem-Ambreen/ea8528e8551363f2de33797ece00000179b99956',
    'https://www.semanticscholar.org/paper/Lens-Clarity-and-Visual-Fatigue-in-Children%3A-The-of-Tushe/a4177c48d90f6d51fc8cd6fd8177f3e648b550dd',
                      ]

display_results(googlescholarurl)
display_results(semanticscholarurl)

#,Result URL
1,https://www.theseus.fi/bitstream/handle/10024/896088/Laine_Jere.pdf?sequence=2
2,https://www.mdpi.com/1660-4601/22/4/609
3,https://personales.upv.es/thinkmind/dl/conferences/achi/achi_2024/achi_2024...
4,https://publications.aston.ac.uk/id/eprint/47842/
5,https://www.tandfonline.com/doi/abs/10.1080/10447318.2024.2305982
6,https://repository.stcloudstate.edu/edad_etds/64/
7,https://www.utupub.fi/bitstream/handle/10024/176173/Virtanen_Julius_opinnay...
8,https://ieeexplore.ieee.org/abstract/document/9089640
9,https://jhsmr.org/index.php/jhsmr/article/view/1095
10,https://link.springer.com/chapter/10.1007/978-3-031-35392-5_10


#,Result URL
1,https://www.semanticscholar.org/paper/Immediate-Effects-of-Light-Mode-and-D...
2,https://www.semanticscholar.org/paper/DarkModeNotifier%3A-Automatic-Dark-Mo...
3,https://www.semanticscholar.org/paper/An-Exploration-of-Effects-of-Dark-Mod...
4,https://www.semanticscholar.org/paper/The-effects-of-lutein-zeaxanthin-(Lut...
5,https://www.semanticscholar.org/paper/Blink-Mate%3A-A-Smart-Screen-Control-...
6,https://www.semanticscholar.org/paper/Interventions-to-prevent-visual-fatig...
7,https://www.semanticscholar.org/paper/Clinical-Efficacy-of-Ayurvedic-Manage...
8,https://www.semanticscholar.org/paper/Astaxanthin-(AstaReal%C2%AE)-Improved...
9,https://www.semanticscholar.org/paper/THE-EFFECTS-OF-BLUE-LIGHT-BLOCKING-GL...
10,https://www.semanticscholar.org/paper/Lens-Clarity-and-Visual-Fatigue-in-Ch...


### Task 3

Then, find a fellow student who will **independently**
assess the results as "relevant" or "not relevant" using the protocol that you
have defined above, and also help (at least) one other student for his/her
assessment. Write down their names here:

**Name of the student who assesses my results:** Dan Gavriluta

**Name of the student who I help to assess his/her/their results:** Dan Gavriluta

Show to the other assessor everything you have written down above for Tasks 1 and 2 (and you might also want to give him/her the PDFs you got for these papers to simplify the process).

You as assessors need to stick to the protocol you made in Task 1 and should not discuss with each other, especially when you doubt whether a result is relevant or not. Write down your assessments as lists of relevance values, as introduced above, and make sure they correctly map to the URLs by displaying them together with the `display_results` function.

To avoid problems with extreme results, mark in each list at least one paper as 'relevant' and at least one paper as 'not relevant'. That is, if all papers seem relevant, mark the one that seems least relevant 'not relevant', and conversely, if none of the papers seem relevant, mark the one that seems a bit more relevant than the others as 'relevant'.

In [21]:
# Assessment 1 from me:
googleAssessment1 = [1,1,1,0,0,0,1,1,1,1]
semanticAssessment1 = [1,1,1,0,0,0,0,0,0,0]

# Assessment 2 from my fellow student:
googleAssessment2 = [1, 1, 1, 1, 1, 0, 1, 1, 1, 1]
semanticAssessment2 = [1, 0, 1, 0, 0, 0, 0, 0, 0, 0]

  
# first search engine assessment
display_results(googlescholarurl, googleAssessment1, googleAssessment2)

# second search engine assessment
display_results(semanticscholarurl, semanticAssessment1, semanticAssessment2)


#,Result URL,Assessment 1,Assessment 2
1,https://www.theseus.fi/bitstream/handle/10024/896088/Laine_Jere.pdf?sequence=2,Relevant,Relevant
2,https://www.mdpi.com/1660-4601/22/4/609,Relevant,Relevant
3,https://personales.upv.es/thinkmind/dl/conferences/achi/achi_2024/achi_2024...,Relevant,Relevant
4,https://publications.aston.ac.uk/id/eprint/47842/,Not relevant,Relevant
5,https://www.tandfonline.com/doi/abs/10.1080/10447318.2024.2305982,Not relevant,Relevant
6,https://repository.stcloudstate.edu/edad_etds/64/,Not relevant,Not relevant
7,https://www.utupub.fi/bitstream/handle/10024/176173/Virtanen_Julius_opinnay...,Relevant,Relevant
8,https://ieeexplore.ieee.org/abstract/document/9089640,Relevant,Relevant
9,https://jhsmr.org/index.php/jhsmr/article/view/1095,Relevant,Relevant
10,https://link.springer.com/chapter/10.1007/978-3-031-35392-5_10,Relevant,Relevant


#,Result URL,Assessment 1,Assessment 2
1,https://www.semanticscholar.org/paper/Immediate-Effects-of-Light-Mode-and-D...,Relevant,Relevant
2,https://www.semanticscholar.org/paper/DarkModeNotifier%3A-Automatic-Dark-Mo...,Relevant,Not relevant
3,https://www.semanticscholar.org/paper/An-Exploration-of-Effects-of-Dark-Mod...,Relevant,Relevant
4,https://www.semanticscholar.org/paper/The-effects-of-lutein-zeaxanthin-(Lut...,Not relevant,Not relevant
5,https://www.semanticscholar.org/paper/Blink-Mate%3A-A-Smart-Screen-Control-...,Not relevant,Not relevant
6,https://www.semanticscholar.org/paper/Interventions-to-prevent-visual-fatig...,Not relevant,Not relevant
7,https://www.semanticscholar.org/paper/Clinical-Efficacy-of-Ayurvedic-Manage...,Not relevant,Not relevant
8,https://www.semanticscholar.org/paper/Astaxanthin-(AstaReal%C2%AE)-Improved...,Not relevant,Not relevant
9,https://www.semanticscholar.org/paper/THE-EFFECTS-OF-BLUE-LIGHT-BLOCKING-GL...,Not relevant,Not relevant
10,https://www.semanticscholar.org/paper/Lens-Clarity-and-Visual-Fatigue-in-Ch...,Not relevant,Not relevant


### Task 4

Compute Cohen's kappa to quantify how much the two assessors agreed. Use the function `cohen_kappa_score` demonstrated above to calculate two times the inter-annotator agreement (once for each of the two search engines), and print out the results.

In [ ]:
print("Google scholar score:", cohen_kappa_score(googleAssessment1, googleAssessment2))
print("Semantic scholar score:", cohen_kappa_score(semanticAssessment1, semanticAssessment2))



Google scholar score: 0.41176470588235303
Semantic scholar score: 0.736842105263158


Explain whether the agreement can be considered high or not, based on the interpretation table on [this Wikipedia page](https://en.wikipedia.org/wiki/Fleiss'_kappa#Interpretation) (this Wikipedia page is about a different type of kappa but the interpretation table can also be used for Cohen's kappa).

**Answer:** 

*0.41 = Moderate agreement*

*0.73 = Substantial agreement*

 Using the interpretation table from the wiki page which groups kappa scores into levels such as slight, fair, moderate, substantial, and almost perfect agreement. The Google Scholar kappa score was 0.41 which falls into the moderate agreement range. This is not considered high because as assessors we still disagreed on a couple documents. With only 10 items, even two disagreements can noticeably lower the score and thus the assessments were not consistent enough to reach the substantial level. The Semantic Scholar kappa score was 0.7368, which fits into the substantial agreement category. This is considered high because the assessors agreed on almost all documents, with very few disagreements. So overall, for Google Scholar we show only moderate agreement, while Semantic Scholar shows high agreement.

### Task 5

Define a function called `precision_at_n` that calculates Precision@n as described in the lecture slides, which takes as input an assessment list and a value for _n_ and returns the respective Precision@n value. Run this function to calculate Precision@10 (that is, n=10) on all four assessments (two assessors and two search engines).

In [ ]:
def precision_at_n(assessment, n):
    relevantCount = sum(assessment)            
    return relevantCount / n              

print("Google, Assessor 1 Precision:", precision_at_n(googleAssessment1, 10))
print("Google, Assessor 2 Precision:", precision_at_n(googleAssessment2, 10))
print("Semantic, Assessor 1 Precision:", precision_at_n(semanticAssessment1, 10))
print("Semantic, Assessor 2 Precision:", precision_at_n(semanticAssessment2, 10))

Google, Assessor 1 Precision: 0.7
Google, Assessor 2 Precision: 0.9
Semantic, Assessor 1 Precision: 0.3
Semantic, Assessor 2 Precision: 0.2


Explain what these specific Precision@10 results tell us (or don't tell us) about the quality of the two search engines for your particular information need. You can also refer to the results of Task 4 if necessary.

**Answer:** 
The Precision@10 results show that for Google Scholar both of us found a relatively high proportion of relevant documents in the top 10 (0.7 and 0.9) which suggests that it retrieves more relevant results for our information need overall. As for with Semantic Scholar, the precision values were much lower (0.3 and 0.2) which indicates that fewer relevant documents appeared among the top 10. However, precision alone does not capture consistency between assessors which is shown in task 4. Our agreement was moderate for Google Scholar and substantial for Semantic Scholar even tho Google Scholar returned more relevant results.

# Submission

Submit the answers to the assignment via Canvas as a modified version of this Notebook file (file with `.ipynb` extension) that includes your code and your answers.

Before submitting, restart the kernel and re-run the complete code (**Kernel > Restart & Run All**), and then check whether your assignment code still works as expected.

Don't forget to add your name, and remember that the assignments have to be done **individually**, and that code sharing or copying are **strictly forbidden** and will be punished.